# Opvullen FactBlueBike

Zal elke 30 minuten uitgevoerd worden

In [2]:
import pandas as pd
import sys
from pathlib import Path
from datetime import datetime, timedelta

ROOT = Path().resolve().parents[1]
sys.path.append(str(ROOT))

# from DWH.connection.connect import getData, text

In [2]:
def get_date_key(dt: datetime) -> int:
    return int(dt.strftime('%Y%m%d'))


def get_time_key(dt: datetime) -> int:
    return int(f"{dt.hour:02d}{dt.minute:02d}")

In [3]:
NOW      = datetime.now() - timedelta(days=1)
DATE_KEY = get_date_key(NOW)
TIME_KEY = get_time_key(NOW)
datetime.day

print(f"Fetch gestart: {NOW.strftime('%Y-%m-%d %H:%M')} → DateKey={DATE_KEY}, TimeKey={TIME_KEY}")

try:
    df_fact = pd.read_json("https://api.blue-bike.be/v4/location/website")
except Exception as e:
    print(f"API call mislukt: {e}")

df_fact = df_fact[['id', 'total_bikes_available', 'e_bikes_available', 'blue_bikes_available', 'max_capacity', 'bikes_defect']]
df_fact.rename(columns={
    'id' : 'BlueBikeStationKey',
    'total_bikes_available' : 'TotalBikesAvailable',
    'e_bikes_available' : 'EBikesAvailable',
    'blue_bikes_available' : 'BlueBikesAvailable',
    'max_capacity' : 'MaxCapacity',
    'bikes_defect' : 'BikesDefect'
}, inplace=True)

# 'isin' zou altijd moeten kloppen behalve als er een nieuwe bluebike locatie is en de dim is niet geupdate
df_dim = getData(query=text("""SELECT * FROM DimBlueBikeStation"""))
df_fact = df_fact[df_fact['BlueBikeStationKey'].isin(df_dim['BlueBikeStationKey'])]

df_fact['DateKey'] = DATE_KEY
df_fact['TimeKey'] = TIME_KEY

print(f"{len(df_fact)} rijen klaar voor FactBlueBike")



Fetch gestart: 2026-03-03 09:25 → DateKey=20260303, TimeKey=925
0 rijen klaar voor FactBlueBike


In [ ]:
# result = loadIN(df=df_fact, table='FactBlueBike')
# print(f"({NOW.strftime('%H:%M')}) FactBlueBike geladen: {result}")

(09:25) FactBlueBike geladen: data has been loaded into table FactBlueBike in database DEPI within the localhost server


In [ ]:
df_bluebike = pd.read_json("https://api.blue-bike.be/pub/location")
df_bluebike = df_bluebike[['id', 'bikes_available', 'bikes_in_use']]
df_bluebike.rename(columns={
    'id' : 'BlueBikeStationKey',
    'bikes_available' : 'AvailableAmount',
    'bikes_in_use' : 'AmountInUse',
}, inplace=True)
try:
    df_fact = pd.read_json("https://api.blue-bike.be/v4/location/website")
except Exception as e:
    print(f"API call mislukt: {e}")

df_fact = df_fact[['id', 'total_bikes_available', 'e_bikes_available', 'blue_bikes_available', 'max_capacity', 'bikes_defect']]
df_fact.rename(columns={
    'id' : 'BlueBikeStationKey',
    'total_bikes_available' : 'TotalBikesAvailable',
    'e_bikes_available' : 'EBikesAvailable',
    'blue_bikes_available' : 'BlueBikesAvailable',
    'max_capacity' : 'MaxCapacity',
    'bikes_defect' : 'BikesDefect'
}, inplace=True)
# print(df_bluebike.info())
# print(df_fact.info())

df_merged = df_bluebike.merge(df_fact, on='BlueBikeStationKey', how='outer', indicator=True)

# Welke key ontbreekt in welke df?
print("Ontbrekend in df_bluebike:")
print(df_merged[df_merged['_merge'] == 'right_only'])

print("Ontbrekend in df_fact:")
print(df_merged[df_merged['_merge'] == 'left_only'])

# Waar verschilt AvailableAmount van TotalBikesAvailable?
df_both = df_merged[df_merged['_merge'] == 'both'].copy()
df_verschil = df_both[df_both['AvailableAmount'] != df_both['TotalBikesAvailable']]
print(f"\nAantal rijen waar AvailableAmount ≠ TotalBikesAvailable: {len(df_verschil)}")
print(df_verschil[['BlueBikeStationKey', 'AvailableAmount', 'TotalBikesAvailable']])

# df_merged2 = df_bluebike.merge(df_fact, on='BlueBikeStationKey', how='inner')
# df_merged2['verwacht_inUse'] = df_merged2['MaxCapacity'] - df_merged2['BlueBikesAvailable'] - df_merged2['EBikesAvailable']
# df_afwijking = df_merged2[df_merged2['verwacht_inUse'] != df_merged2['AmountInUse']]

# print(f"Aantal afwijkingen: {len(df_afwijking)}")
# print(df_afwijking[['BlueBikeStationKey', 'MaxCapacity', 'BlueBikesAvailable', 'EBikesAvailable', 'verwacht_inUse', 'AmountInUse']])

# df_negatief = df_merged2[df_merged2['verwacht_inUse'] < 0]
# print(f"Aantal negatief: {len(df_negatief)}")
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 200)
# print(df_negatief[['BlueBikeStationKey', 'MaxCapacity', 'BlueBikesAvailable', 'EBikesAvailable', 'verwacht_inUse', 'AmountInUse']])
# pd.reset_option('display.max_rows')
# pd.reset_option('display.max_columns')
# pd.reset_option('display.width')
# print(df_negatief[['MaxCapacity', 'BlueBikesAvailable', 'EBikesAvailable', 'verwacht_inUse']])

Ontbrekend in df_bluebike:
     BlueBikeStationKey  AvailableAmount  AmountInUse  TotalBikesAvailable  \
213                 326              NaN          NaN                    0   

     EBikesAvailable  BlueBikesAvailable  MaxCapacity  BikesDefect      _merge  
213                0                   0            6            0  right_only  
Ontbrekend in df_fact:
Empty DataFrame
Columns: [BlueBikeStationKey, AvailableAmount, AmountInUse, TotalBikesAvailable, EBikesAvailable, BlueBikesAvailable, MaxCapacity, BikesDefect, _merge]
Index: []

Aantal rijen waar AvailableAmount ≠ TotalBikesAvailable: 0
Empty DataFrame
Columns: [BlueBikeStationKey, AvailableAmount, TotalBikesAvailable]
Index: []
